# Querying Museum Data

This notebook connects to the museum database and conducts simple analysis of the data.

The tasks of this notebook is to:
* Connect to the remote database
* Extract useful information fro the stakeholders using SQL queries
* Write clean, maintainable code


**Analysis goal**: help stakeholders make informed choices about exhibition planning and visitor safety, based on real data.

Stakeholders:

1. Angela Millay: Exhibitions Manager
    * Identify successful exhibitions
2. Rita Pelkman: Head of Security & Visitor Safety
    * staff patrol routes
    * Potential hazards, sites, times E.g where customers are most likely to ask for assistance
    * Whether these are emergencies or just need assistance

# Imports

In [71]:

from dotenv import dotenv_values
from psycopg2 import connect
import pandas as pd
import altair as alt
env_values = dotenv_values()

# Analysis

In [72]:
def get_db_connection():
    """Get a connection to the PostgreSQL database
    using credentials from .env file."""
    env_values = dotenv_values()
    return connect(host=env_values["DB_HOST"], dbname="museum",
                   user=env_values["DB_USER"], password=env_values["DB_PASSWORD"])


conn = get_db_connection()

In [73]:
pd.read_sql("SELECT * FROM button_type LIMIT 5", conn)

/var/folders/5s/0sgwq2vj7k34s76fs56pym840000gn/T/ipykernel_54414/616065681.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT * FROM button_type LIMIT 5", conn)


,val,type,description,button_id
0,-1,0.0,Assistance,0
1,-1,1.0,Emergency,1
2,0,NaN,Terrible,2
3,1,NaN,Bad,3
4,2,NaN,Neutral,4


# Angela - Exhibition Ratings

Looking at the ratings of the different exhibitions:

- We will exclude the emergencies / assistance called and just looked at emotional scale

In [74]:
query = """
SELECT e.name, e.floor,
       AVG(bt.val) as avg_rating,
       COUNT(*) as num_ratings
FROM exhibition e
JOIN visitor_interactions vi ON e.id = vi.site
JOIN button_type bt ON vi.button_id = bt.button_id
WHERE bt.val >= 0
GROUP BY e.name, e.floor
ORDER BY avg_rating DESC
"""
ratings_df = pd.read_sql(query, conn)
ratings_df

/var/folders/5s/0sgwq2vj7k34s76fs56pym840000gn/T/ipykernel_54414/2734606023.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ratings_df = pd.read_sql(query, conn)


,name,floor,avg_rating,num_ratings
0,Cetacean Sensations,1,3.000000,117
1,Adaptation,-1,2.051020,98
2,Measureless to Man,1,1.948718,78
3,The Crenshaw Collection,2,1.488636,88
4,Our Polluted World,3,1.196970,132


In [79]:
alt.Chart(ratings_df).mark_bar(color='steelblue').encode(
    x=alt.X('avg_rating:Q', title='Average Rating (0-4)',
            scale=alt.Scale(domain=[0, 4])),
    y=alt.Y('name:N', sort='-x', title='Exhibition'),
    tooltip=['name', 'avg_rating', 'num_ratings'],
    color=alt.Color('name:N', legend=None),
).properties(
    title='Exhibition Success: Average Ratings',
    width=600,
    height=400
).configure_view(strokeWidth=0)

alt.Chart(...)

In [76]:
query = """
WITH ranked_ratings AS (
    SELECT e.name,
           bt.description,
           COUNT(*) as count,
           ROW_NUMBER() OVER (PARTITION BY e.name ORDER BY COUNT(*) DESC) as rank
    FROM exhibition e
    JOIN visitor_interactions vi ON e.id = vi.site
    JOIN button_type bt ON vi.button_id = bt.button_id
    WHERE bt.val >= 0
    GROUP BY e.name, bt.description
)
SELECT name, description, count
FROM ranked_ratings
WHERE rank = 1
ORDER BY count DESC
"""
most_pressed_df = pd.read_sql(query, conn)
alt.Chart(most_pressed_df).mark_bar(color='#2ca02c').encode(
    x=alt.X('count:Q', title='Number of Times Pressed'),
    y=alt.Y('name:N', sort='-x', title='Exhibition'),
    color=alt.Color('description:N'),
    tooltip=['name', 'description', 'count']
).properties(
    title='Most Common Rating Per Exhibition',
    width=600
)

/var/folders/5s/0sgwq2vj7k34s76fs56pym840000gn/T/ipykernel_54414/1455467009.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  most_pressed_df = pd.read_sql(query, conn)


alt.Chart(...)

In [77]:
query = """
SELECT bt.description, COUNT(*) as total_count
FROM visitor_interactions vi
JOIN button_type bt ON vi.button_id = bt.button_id
GROUP BY bt.description
ORDER BY total_count DESC
"""
all_buttons_df = pd.read_sql(query, conn)

alt.Chart(all_buttons_df).mark_bar(color='steelblue').encode(
    x=alt.X('description:N', title='Button Type', sort='-y'),
    y=alt.Y('total_count:Q', title='Total Presses'),
    tooltip=['description', 'total_count'],
    color=alt.Color('description', legend=None)
).properties(
    title='All Button Presses by Type',
    width=600,
    height=300
)

/var/folders/5s/0sgwq2vj7k34s76fs56pym840000gn/T/ipykernel_54414/223684321.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  all_buttons_df = pd.read_sql(query, conn)


alt.Chart(...)

# Rita: Emergency vs Assistance counts

In [33]:
query = """
SELECT e.name, e.floor,
       SUM(CASE WHEN bt.description = 'Emergency' THEN 1 ELSE 0 END) as emergencies,
       SUM(CASE WHEN bt.description = 'Assistance' THEN 1 ELSE 0 END) as assistance
FROM exhibition e
JOIN visitor_interactions vi ON e.id = vi.site
JOIN button_type bt ON vi.button_id = bt.button_id
WHERE bt.description IN ('Emergency', 'Assistance')
GROUP BY e.name, e.floor
ORDER BY emergencies DESC
"""
safety_df = pd.read_sql(query, conn)
safety_df

/var/folders/5s/0sgwq2vj7k34s76fs56pym840000gn/T/ipykernel_54414/3906388710.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  safety_df = pd.read_sql(query, conn)


,name,floor,emergencies,assistance
0,Adaptation,-1,0,1
1,Cetacean Sensations,1,0,7
2,Measureless to Man,1,0,4
3,Our Polluted World,3,0,5
4,The Crenshaw Collection,2,0,4


In [49]:
query = """
SELECT e.name, bt.description, COUNT(*) as count
FROM exhibition e
JOIN visitor_interactions vi ON e.id = vi.site
JOIN button_type bt ON vi.button_id = bt.button_id
WHERE bt.description IN ('Emergency', 'Assistance')
GROUP BY e.name, bt.description
"""
safety_df = pd.read_sql(query, conn)
safety_df

alt.Chart(safety_df).mark_bar().encode(
    x=alt.X('name:N', title='Exhibition', sort='-y'),
    y=alt.Y('count:Q', title='Number of Incidents'),
    color=alt.Color('description:N',
                    scale=alt.Scale(domain=['Emergency', 'Assistance'],
                                    range=['#d62728', '#ff7f0e']),
                    legend=None),
    xOffset='description:N',
    tooltip=['name', 'description', 'count']
).properties(
    title='Safety Incidents by Exhibition',
    width=600
)

/var/folders/5s/0sgwq2vj7k34s76fs56pym840000gn/T/ipykernel_54414/2888201793.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  safety_df = pd.read_sql(query, conn)


alt.Chart(...)

In [61]:
query = """
SELECT vi.at as timestamp,
       bt.description,
       bt.val
FROM visitor_interactions vi
JOIN button_type bt ON vi.button_id = bt.button_id
ORDER BY vi.at
"""
all_interactions_df = pd.read_sql(query, conn)
all_interactions_df['timestamp'] = all_interactions_df['timestamp'].astype(str)

alt.Chart(all_interactions_df).mark_circle(size=60).encode(
    x=alt.X('timestamp:T', title='Time'),
    y=alt.Y('description:N', title='Interaction Type',
            sort=alt.EncodingSortField('val', order='descending')),
    color=alt.Color('description:N', legend=None),
    tooltip=['timestamp', 'description']
).properties(
    title='All Visitor Interactions Over Time',
    width=700,
    height=400
)

/var/folders/5s/0sgwq2vj7k34s76fs56pym840000gn/T/ipykernel_54414/1304927436.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  all_interactions_df = pd.read_sql(query, conn)


alt.Chart(...)

In [51]:
conn.rollback()  # Rollback any open transaction

In [ ]:
conn.close()